In [1]:
from src.model import XGemma3ForCausalLM

In [2]:
import torch

In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [5]:
model = XGemma3ForCausalLM.from_pretrained("brimmann2/xgemma3-1b-v1", torch_dtype = torch.bfloat16,low_cpu_mem_usage = True)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("brimmann2/xgemma3-1b-v1",add_eos_token=False,use_fast=False,padding_side='left')

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

In [7]:
XRAG_TOKEN = "<xRAG>"
model.set_xrag_token_id(tokenizer.convert_tokens_to_ids(XRAG_TOKEN))
from src.model import SFR
device = "cuda"

In [8]:
retriever_name_or_path = "Salesforce/SFR-Embedding-Mistral"
retriever = SFR.from_pretrained(retriever_name_or_path,torch_dtype = torch.bfloat16).eval().to(device)
retriever_tokenizer = AutoTokenizer.from_pretrained(retriever_name_or_path)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
doc = "In January 2009 President Barack Obama restored US funding to UNFPA, saying in a public statement that he would \"look forward to working with Congress to restore US financial support for the UN Population Fund. By resuming funding to UNFPA, the US will be joining 180 other donor nations working collaboratively to reduce poverty, improve the health of women and children, prevent HIV/AIDS and provide family planning assistance to women in 154 countries."

In [10]:
retriever_input = retriever_tokenizer(doc,max_length=180,padding=True,truncation=True,return_tensors='pt').to(device)
with torch.no_grad():
    doc_embeds = retriever.get_doc_embedding(input_ids=retriever_input.input_ids,attention_mask=retriever_input.attention_mask)
print(doc_embeds.shape)

torch.Size([1, 4096])


In [11]:
## 4. concate the doc and query in a template
prompt = """<start_of_turn>user
Background: <xRAG> is a paraphrase of what?<end_of_turn>
<start_of_turn>model"""

print(prompt)

model.to(device)

<start_of_turn>user
Background: <xRAG> is a paraphrase of what?<end_of_turn>
<start_of_turn>model


XGemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262146, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), 

In [12]:
input_ids = tokenizer(prompt,return_tensors='pt').input_ids.to(device)
generated_output = model.generate(
        input_ids = input_ids,
        do_sample=False,
        max_new_tokens=100,
        pad_token_id=tokenizer.convert_tokens_to_ids("<pad>"),
        retrieval_embeds = doc_embeds.unsqueeze(0),
    )
result = tokenizer.batch_decode(generated_output,skip_special_tokens=True)[0]
print(result)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



The United States Department of State has been working with the UN to promote the global health of the world's most vulnerable populations. The Department has been working to provide the world's most vulnerable populations with the resources they need to combat the spread of HIV/AIDS, tuberculosis, malaria, and other diseases. The Department has also been working to provide the world's most vulnerable populations with the resources they need to combat the spread of HIV/AIDS, tuberculosis, malaria, and other diseases.
